<a href="https://colab.research.google.com/github/adljna/ProjectA-Group3-KematianAliKhamenei/blob/main/Article%20Link%20Scrapping/CNN/1-CNN-ScrappingLink.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Indonesia Scrapping
**Query:** `kematian ali khamenei`  
**Output:** `scraping_cnn_100pages.csv`

In [ ]:
# ─── Cell 1: Install dependencies ───────────────────────────────────────────
!pip install -q selenium webdriver-manager beautifulsoup4 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.5 MB/s eta 0:00:00


In [ ]:
# ─── Cell 2: Install Google Chrome (Colab environment) ──────────────────────
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update -q
!apt-get install -y -q google-chrome-stable
print('✅ Chrome berhasil diinstall')

OK
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:11 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,218 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,237 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,601 kB]
Get:14 http://arch

In [ ]:
# ─── Cell 3: Import library & inisialisasi WebDriver ────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
import os

# Konfigurasi Chrome headless
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--no-zygote')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/120.0.0.0 Safari/537.36'
)
chrome_options.binary_location = '/usr/bin/google-chrome'

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)
driver.set_page_load_timeout(30)
print('✅ WebDriver siap digunakan')

✅ WebDriver siap digunakan


In [ ]:
# ─── Cell 4: Fungsi scraping satu halaman ───────────────────────────────────

def scrape_page(driver, url, page_num, wait_sec=5, max_retry=3):
    """
    Scrape satu halaman hasil pencarian CNN Indonesia.
    Mengembalikan list of dict dengan kunci:
      Judul, Link, Kategori, Tanggal_Upload
    """
    for attempt in range(1, max_retry + 1):
        try:
            driver.get(url)
            time.sleep(wait_sec)          # tunggu JS render
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            break
        except Exception as e:
            print(f'  ⚠️  Halaman {page_num} percobaan {attempt} gagal: {e}')
            if attempt == max_retry:
                print(f'  ❌ Halaman {page_num} dilewati setelah {max_retry}x retry.')
                return []
            time.sleep(wait_sec * 2)

    scraped_data = []

    # ── Selector utama (sesuai struktur HTML CNN Indonesia) ─────────────────
    # <article class="flex-grow ..."> berisi setiap kartu berita
    articles = soup.find_all('article', class_=lambda c: c and 'flex-grow' in c)

    if not articles:
        # fallback: cari semua <article> saja
        articles = soup.find_all('article')

    for article in articles:
        # ── Link ──────────────────────────────────────────────────────────
        link = ''
        link_el = article.find('a', class_=lambda c: c and 'flex' in c)
        if link_el and link_el.get('href'):
            link = link_el['href'].strip()
        else:
            link_el = article.find('a', href=True)
            if link_el:
                link = link_el['href'].strip()

        # Pastikan link absolut
        if link and not link.startswith('http'):
            link = 'https://www.cnnindonesia.com' + link

        # ── Judul ─────────────────────────────────────────────────────────
        title = ''
        for tag in ['h2', 'h3', 'h1']:
            title_el = article.find(tag)
            if title_el:
                title = title_el.get_text(strip=True)
                break

        # ── Kategori ──────────────────────────────────────────────────────
        category = ''
        # Kelas umum CNN Indonesia untuk label kategori
        cat_el = article.find('span', class_=lambda c: c and 'cnn_red' in c)
        if cat_el:
            category = cat_el.get_text(strip=True)
        else:
            # fallback: cari span xs pertama
            cat_el = article.find('span', class_=lambda c: c and 'text-xs' in c)
            if cat_el:
                category = cat_el.get_text(strip=True)

        # ── Tanggal ───────────────────────────────────────────────────────
        date = ''
        date_el = article.find('span', class_=lambda c: c and 'cnn_black_light3' in c)
        if date_el:
            date = date_el.get_text(strip=True).replace('•', '').strip()
        else:
            # fallback: cari semua span dan ambil yang berisi angka tahun
            for span in article.find_all('span'):
                txt = span.get_text(strip=True)
                if re.search(r'\d{4}', txt):
                    date = txt.replace('•', '').strip()
                    break

        # Hanya simpan jika minimal ada judul atau link
        if title or link:
            scraped_data.append({
                'Judul'         : title,
                'Link'          : link,
                'Kategori'      : category,
                'Tanggal_Upload': date
            })

    return scraped_data


print('✅ Fungsi scrape_page siap')

✅ Fungsi scrape_page siap


In [ ]:
# ─── Cell 5: Loop utama — scraping 100 halaman ──────────────────────────────

QUERY        = 'kematian%20ali%20khamenei'   # URL-encoded query
# Halaman 1 tidak pakai parameter &page=, halaman 2+ baru pakai &page=N
URL_PAGE1    = f'https://www.cnnindonesia.com/search/?query={QUERY}'
URL_PAGE_N   = 'https://www.cnnindonesia.com/search/?query={query}&page={page}'
TOTAL_PAGES  = 10
WAIT_SEC     = 5      # detik tunggu setelah get()
SLEEP_BETWEEN= 2      # detik jeda antar halaman
OUTPUT_RAW   = 'scraping_cnn_raw.csv'

all_rows = []

for page in range(1, TOTAL_PAGES + 1):
    # Halaman 1: tanpa &page=  |  Halaman 2-100: dengan &page=N
    if page == 1:
        url = URL_PAGE1
    else:
        url = URL_PAGE_N.format(query=QUERY, page=page)
    print(f'📄 Scraping halaman {page:>3}/{TOTAL_PAGES} | URL: {url}')

    rows = scrape_page(driver, url, page, wait_sec=WAIT_SEC)

    if rows:
        all_rows.extend(rows)
        print(f'   ✅ {len(rows)} artikel ditemukan  (total: {len(all_rows)})')
    else:
        print('   ⚠️  Tidak ada artikel / halaman kosong')

    # Simpan checkpoint setiap 10 halaman
    if page % 10 == 0:
        pd.DataFrame(all_rows).to_csv(OUTPUT_RAW, index=False, encoding='utf-8')
        print(f'  💾 Checkpoint disimpan ke {OUTPUT_RAW}')

    time.sleep(SLEEP_BETWEEN)

# Simpan hasil akhir raw
df_raw = pd.DataFrame(all_rows)
df_raw.to_csv(OUTPUT_RAW, index=False, encoding='utf-8')

print(f'\n🎉 Selesai! Total baris (raw): {len(df_raw)}')
print(df_raw.head(3))

📄 Scraping halaman   1/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei
   ✅ 10 artikel ditemukan  (total: 10)
📄 Scraping halaman   2/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=2
   ✅ 10 artikel ditemukan  (total: 20)
📄 Scraping halaman   3/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=3
   ✅ 10 artikel ditemukan  (total: 30)
📄 Scraping halaman   4/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=4
   ✅ 10 artikel ditemukan  (total: 40)
📄 Scraping halaman   5/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=5
   ✅ 10 artikel ditemukan  (total: 50)
📄 Scraping halaman   6/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=6
   ✅ 10 artikel ditemukan  (total: 60)
📄 Scraping halaman   7/10 | URL: https://www.cnnindonesia.com/search/?query=kematian%20ali%20khamenei&page=7
   ✅ 10 ar

In [ ]:
# ─── Cell 6: Tutup browser ──────────────────────────────────────────────────
driver.quit()
print('✅ Browser ditutup')

✅ Browser ditutup


In [ ]:
# ─── Cell 7: Pembersihan data ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re

df = pd.read_csv('scraping_cnn_raw.csv', encoding='utf-8')
print(f'Baris sebelum cleaning: {len(df)}')

# ── Fungsi pembersihan nilai ─────────────────────────────────────────────────
def clean_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return np.nan if x in ('', '#', 'nan', 'None') else x

df = df.applymap(clean_value)

# ── Hapus baris kosong semua kolom ───────────────────────────────────────────
df = df.dropna(how='all')

# ── Hapus baris tanpa Judul DAN tanpa Link ───────────────────────────────────
df = df[~(df['Judul'].isna() & df['Link'].isna())]

# ── Filter link hanya yang benar-benar URL CNN Indonesia ─────────────────────
df = df[df['Link'].str.startswith('https://www.cnnindonesia.com/', na=False)]

# ── Hapus duplikat berdasarkan Link (kolom paling unik) ──────────────────────
df = df.drop_duplicates(subset=['Link'], keep='first')

# ── Bersihkan kolom Tanggal_Upload (hilangkan bullet & whitespace ganda) ──────
df['Tanggal_Upload'] = (
    df['Tanggal_Upload']
    .str.replace(r'[•|]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# ── Reset index ───────────────────────────────────────────────────────────────
df = df.reset_index(drop=True)

# ── Simpan hasil bersih ───────────────────────────────────────────────────────
OUTPUT_CLEAN = 'scraping_cnn_100pages_clean.csv'
df.to_csv(OUTPUT_CLEAN, index=False, encoding='utf-8-sig')   # utf-8-sig agar Excel bisa buka

print(f'Baris setelah cleaning  : {len(df)}')
print(f'File disimpan           : {OUTPUT_CLEAN}')
print()
print(df.info())
print()
print(df.head(10).to_string())

Baris sebelum cleaning: 100
Baris setelah cleaning  : 99
File disimpan           : scraping_cnn_100pages_clean.csv

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Judul           99 non-null     object
 1   Link            99 non-null     object
 2   Kategori        99 non-null     object
 3   Tanggal_Upload  99 non-null     object
dtypes: object(4)
memory usage: 3.2+ KB
None

                                                                   Judul                                                                                                                                        Link       Kategori     Tanggal_Upload
0                 Isi Dokumen Fatwa Ali Khamenei Mengharamkan Bom Nuklir                https://www.cnnindonesia.com/internasional/20260428200911-120-1353182/isi-dokumen-fatwa-ali-khamenei-mengharamkan-bom-nuklir  Internasion

/tmp/ipykernel_636/2999827071.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_value)


In [ ]:
# ─── Cell 8: Ringkasan statistik ─────────────────────────────────────────────
print('=== RINGKASAN HASIL SCRAPING ===')
print(f'Total artikel unik : {len(df)}')
print()

print('--- Distribusi Kategori (Top 15) ---')
print(df['Kategori'].value_counts().head(15).to_string())
print()

print('--- Nilai NaN per Kolom ---')
print(df.isna().sum().to_string())
print()

print(f'File akhir: scraping_cnn_100pages_clean.csv')

=== RINGKASAN HASIL SCRAPING ===
Total artikel unik : 99

--- Distribusi Kategori (Top 15) ---
Kategori
Internasional    89
Hiburan           4
Nasional          3
BREAKING NEWS     2
ANALISIS          1

--- Nilai NaN per Kolom ---
Judul             0
Link              0
Kategori          0
Tanggal_Upload    0

File akhir: scraping_cnn_100pages_clean.csv


In [ ]:
# ─── Cell 9: Download file CSV dari Colab ────────────────────────────────────
from google.colab import files
files.download('scraping_cnn_100pages_clean.csv')
files.download('scraping_cnn_raw.csv')   # opsional — data mentah sebelum cleaning

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>